# Script que plotea QLSS de galaxias aisladas (SIG) y galaxias aisladas en vacíos (cruce entre Pann y SIG) (o lo intenta)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import fsolve
from scipy.optimize import curve_fit
from scipy.stats import norm
from typing import List

### Functions

In [ ]:
def multiStr2Num(n: pd.core.series.Series) -> np.array:
    '''
    Converts a list of numbers in str format to float (n) format
    '''
    Num = np.zeros(len(n))
    for i,c in enumerate(n):
        Num[i] = float(c)
    return Num

In [ ]:
def open_txt(fich: str) -> pd.DataFrame:
    '''
    Opens the file and returns a DF.
    (fich) Name of the file to open
    '''
    # Extracts line by line
    with open(fich, 'r') as file:
        L = [lines.split() for lines in file]

    # We get the column names from the page
    column_names_2 = ['name_1','ra','dec','modelMag_r','petroR90_r','z','MAG_u','MAG_g','MAG_r','MAG_i',
                      'MAG_z','stellarmas','name_2,','k','Q_LSS','eta_k','dnn','d5n','f_LSS']
    dfo = pd.DataFrame(L[1:], columns=L[0]) # Create the DF

    df = dfo.copy()
    for c in L[0]:
        df[c] = multiStr2Num(df[c])
    return df


def clean(df: pd.DataFrame, columns: List[str]) -> pd.DataFrame:
    '''
    Reduces the columns to the specified in the parametre columns for a given DF
    '''
    #Verifying if the specified columns are in the DataFrame
    valid_columns = [col for col in columns if col in df.columns]

    if len(valid_columns) != len(columns):
        non_valid_columns = set(columns) - set(valid_columns)
        print(f"Warning: Non existent columns have been written,following will not be included: {non_valid_columns}")

    #Croping the DF only with the valid columns

    df_croped = df[valid_columns]

    return df_croped

In [ ]:

def gaussian(x, a, b, c):
    ''' 
    Gaussian function for later fit
    '''
    return a * np.exp(-(x - b)**2 / (2 * c**2))


### Main

In [ ]:
#Reading the files we are going to use

SIG = open_txt('data/SIG.txt')

Pann_SIG = pd.read_csv('data/Pann_SIG.csv') 



In [ ]:
#Files obtained my merging catalogues together

Pann_SIG_AGN = pd.read_csv('data/Pann_SIG_AGN.csv') #Isolated galaxies in voids with AGN presence
SIG_AGN_walls = pd.read_csv('data/SIG_AGN_walls.csv') # Isolated galaxies in walls with AGN presence

SIG_Walls_ALL = pd.read_csv('data/SIG_Walls_ALL.csv') #DF of isolated galaxies in walls with all their properties, with and without AGN
Pann_SIG_ALL = pd.read_csv('data/Pann_SIG_ALL.csv') #DF of isolated galaxies in voids with all their properties, with and without AGN

> For absent data in the Q_LSS column, DF fills the Nan space with the smallest number (-99), I must filter it to stopping it from bluring the sample (this will be done when necessary)

In [ ]:
cond= SIG['Q_LSS']>-20 

In [ ]:
#ploting a Q_LSS histogram for isolated galaxies

plt.hist(SIG[cond]['Q_LSS'], bins = 50, edgecolor='black') 


In [ ]:
cond2= Pann_SIG['Q_LSS']>-20
plt.hist(Pann_SIG[cond2]['Q_LSS'], bins=50 ,edgecolor='black')



In [ ]:
cond= SIG['Q_LSS']>-20
cond2= Pann_SIG['Q_LSS']>-20

plt.hist(Pann_SIG[cond2]['Q_LSS'], density= 'True', alpha = 0.6, bins=50 ,edgecolor='black', label= 'SIG in voids') #normalised via density order
plt.hist(SIG[cond]['Q_LSS'], density= 'True', bins = 50, alpha = 0.5, edgecolor='black', label= 'SIG') 
plt.legend()
plt.xlabel ('Q_lss')
plt.ylabel('Number of galaxies')
plt.title ('Normalised Q_lss histogram for isolated galaxies and isolated galaxies in voids')

plt.savefig('plots/')


> We see that there is a difference between the Q_LSS value in isolated galaxies in general and isolated galaxies in voids, we then ask ourselves if it is a big enough difference to conclude that isolated galaxies in voids have a higher value of Q_LSS


### In this part of the script I filter the isolated galaxies that are not in voids (hence are in walls) and compare their Q_LSS with those in voids, and I calculate their mean and standard deviation value, to see if I can classify the nuclear activity and/or enviroment of isolated galaxies just by looking at Q_LSS value

>Isolated galaxies are not found in voids, hence an isolated galaxy (in SIG) which is not in a void (in Pann) must be in a wall enviroment

In [ ]:

df1=SIG #the bigger one
df2=Pann_SIG #subsection of the first

In [ ]:
#The df of galaxies in walls will contain the non common data between df1 y df2

df2= df2.rename(columns={'cz':'z'})

df_combined= pd.merge (df1,df2, on= ['ra', 'dec'], how= 'outer', indicator='True') #este df tiene los datos de SIG y Pann_SIG

#Columns only common ro df1 or df2
df_not_common = df_combined[df_combined['True'] != 'both']

# Eliminate the columns that I will not use
SIG_Walls = df_not_common.drop(columns=['True'])

SIG_Walls


In [ ]:
cond3= SIG_Walls['Q_LSS_x']>-10
plt.hist(SIG_Walls[cond3]['Q_LSS_x'], bins=50, density= 'True', edgecolor='black', label= 'SIG in walls' )
plt.legend()
plt.xlabel('Q_lss')
plt.ylabel('Number of galaxies')
plt.title('Normalised Q_lss histogram for isolated galaxies in walls')



In [ ]:

mediana_Pann = Pann_SIG[cond2]['Q_LSS'].median()
mediana_walls = SIG_Walls[cond3]['Q_LSS_x'].median()

round(mediana_walls, 2)

In [ ]:
plt.hist(SIG_Walls[cond3]['Q_LSS_x'], bins=50, density= 'True', edgecolor='black', color='darkviolet', alpha=0.4, label= 'SIG in walls' )
plt.hist(Pann_SIG[cond2]['Q_LSS'], bins=50, density= 'True', color='green', edgecolor='black', alpha=0.4, label= 'SIG in voids' )

#mean and standard deviation value
mean_noPann= np.mean(SIG_Walls[cond3]['Q_LSS_x'])
std_noPann= np.std(SIG_Walls[cond3]['Q_LSS_x'])

mean_Pann= np.mean(Pann_SIG[cond2]['Q_LSS'])
std_Pann= np.std(Pann_SIG[cond2]['Q_LSS'])

mediana_Pann = Pann_SIG[cond2]['Q_LSS'].median()
mediana_walls = SIG_Walls[cond3]['Q_LSS_x'].median()

plt.axvline(mediana_Pann, color='darkgreen', linestyle='--', linewidth = 2, label='Median SIG in voids')
plt.axvline(mediana_walls, color='indigo', linestyle='--', linewidth = 2,label='Median SIG in walls')


plt.legend()
plt.xlabel (r'$Q_{lss}$', fontsize = 13)
plt.yticks(fontsize = 12)
plt.xticks(fontsize = 12)
plt.text(-4, 0.4, r'$-4.91$', color = 'indigo', ha = 'left', size = 12)
plt.text(-7.2, 0.4 , r'$-5.57$', color = 'darkgreen', ha = 'left', size = 12)

plt.savefig('plots/Histogram_QLSS_medians.png')


In [ ]:
plt.hist(SIG_Walls[cond3]['Q_LSS_x'], bins=50, density= 'True', edgecolor='black', color='purple', alpha=0.5, label= 'SIG in walls' )
plt.hist(Pann_SIG[cond2]['Q_LSS'], bins=50, density= 'True', color='green', edgecolor='black', alpha=0.4, label= 'SIG in voids' )

#mean and standard deviation values 
mean_Walls= np.mean(SIG_Walls[cond3]['Q_LSS_x'])
std_Walls= np.std(SIG_Walls[cond3]['Q_LSS_x'])


mean_Pann= np.mean(Pann_SIG[cond2]['Q_LSS'])
std_Pann= np.std(Pann_SIG[cond2]['Q_LSS'])


#representing the obtained data
plt.axvline(mean_Pann, color='red', linestyle='--', label='Media SIG in voids')
plt.axvline(mean_Pann + std_Pann, color='orange', linestyle='--')
plt.axvline(mean_Pann - std_Pann, color='orange', linestyle='--')

plt.axvline(mean_Walls, color='lightblue', linestyle='--', label='Media SIG in walls')
plt.axvline(mean_Walls + std_Walls, color='blue', linestyle='--')
plt.axvline(mean_Walls - std_Walls, color='blue', linestyle='--')


plt.legend()
plt.xlabel ('Q_lss')
plt.ylabel('Number of galaxies')
plt.title('Normalised Q_lss histogram for SIG in voids and SIG in walls')

In [ ]:
mean_Walls, std_Walls


In [ ]:
mean_Pann, std_Pann

### In this part of the script I have fitted the histograms to two Gaussian distributions and calculated their intersection point for each catalogue

In [ ]:

#PANN_SIG
counts_pann, bin_edges_pann = np.histogram(Pann_SIG[cond2]['Q_LSS'], bins=50, density= 'True')
bin_centres_pann = (bin_edges_pann[:-1] + bin_edges_pann[1:]) / 2 #middle points of histogram bins

#initial estimations
p0= [max(counts_pann), np.mean(Pann_SIG[cond2]['Q_LSS']), np.std(Pann_SIG[cond2]['Q_LSS'])]

#curve fit
popt_pann, pcov_pann = curve_fit(gaussian, bin_centres_pann, counts_pann, p0=p0)

#optimal parametres
a_opt_pann, b_opt_pann, c_opt_pann = popt_pann

#generating curve fit values
x_fit_pann= np.linspace(min(bin_centres_pann), max(bin_centres_pann), 500)
y_fit_pann= gaussian(x_fit_pann, *popt_pann)

In [ ]:
#SIG_WALLS
counts, bin_edges = np.histogram(SIG_Walls[cond3]['Q_LSS_x'], bins=50, density= 'True')
bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2 #middle points of histogram bins


#initial estimations
p0= [max(counts), np.mean(SIG_Walls[cond3]['Q_LSS_x']), np.std(SIG_Walls[cond3]['Q_LSS_x'])]

#curve fit
popt, pcov = curve_fit(gaussian, bin_centres, counts, p0=p0)

#optimal parametres
a_opt, b_opt, c_opt = popt

#generating curve fit values
x_fit= np.linspace(min(bin_centres), max(bin_centres), 500)
y_fit= gaussian(x_fit, *popt)

In [ ]:
# Representing histograms and curves
plt.hist(Pann_SIG[cond2]['Q_LSS'], bins=40, density=True, alpha=0.4, color='green', edgecolor='black', label="SIG in voids")
plt.plot(x_fit_pann, y_fit_pann, label="Fit", color="red")

plt.hist(SIG_Walls[cond3]['Q_LSS_x'], bins=40, density=True, alpha=0.4, color='purple', edgecolor='black', label="SIG in walls")
plt.plot(x_fit, y_fit, label="Fit", color="blue")

plt.legend()
plt.xlabel("Q_lss")
plt.ylabel("Number of galaxies")
plt.title("Gaussian fit to the Q_lss histogram for SIG in voids and SIG in walls")
plt.show()


In [ ]:
#Intersection point between gaussian fits

xx = np.linspace(-9, -2, 1000)

ft = gaussian(xx, a_opt, b_opt, c_opt)
gt = gaussian(xx, a_opt_pann, b_opt_pann, c_opt_pann)

idx = np.argwhere(np.diff(np.sign(ft - gt)) != 0).reshape(-1) #this is a list from which I'm only interested in the first value 

print(xx[idx][0], ft[idx][0]) #coordinates of the intersection point

In [ ]:
#Plotting gaussians and interactions

#histograms 
plt.hist(Pann_SIG[cond2]['Q_LSS'], bins=50, density=True, alpha=0.6, label="Pann_SIG ")
plt.hist(SIG_Walls[cond3]['Q_LSS_x'], bins=50, density=True, alpha=0.6, label="SIG_Walls")

#fits
plt.plot(x_fit_pann, y_fit_pann, label="Gaussian fit Pann_SIG", color="blue")
plt.plot(x_fit, y_fit, label="Gaussian fit SIG_Walls", color="red")

#intersecciones
plt.scatter(xx[idx][0], ft[idx][0], color='black', label='Intersections')


plt.legend()
plt.xlabel("Q_LSS")
plt.ylabel("Number of galaxies")
plt.title("Points of intersections between Gaussian fits")
plt.show()


>There is not enough evidente to conclude that there is a single value of Q_LSS that allows us to caracterize the enviroment of the galaxy :(

### In this part of the script I try to see the differences between isolated galaxies with and without AGN presence to see if nuclear activity plays some kind of role in isolation

In [ ]:
#Scatter plot 

x = Pann_SIG[cond2]['effr_frac']
y = Pann_SIG[cond2]['Q_LSS']

x_A = Pann_SIG_AGN['effr_frac']
y_A = Pann_SIG_AGN['Q_LSS']



plt.scatter(x, y, color='blue', marker='o', label='Without AGN')
plt.scatter(x_A, y_A, color='r', marker='o', label='With AGN' )


plt.ylabel('Q_lss')
plt.xlabel('Effective radius')
plt.title('Effective radius vs Q_LSS for isolated galaxies in voids')
plt.legend()
plt.grid(True)

plt.show


In [ ]:


x = Pann_SIG[cond2]['effr_frac']
y = Pann_SIG[cond2]['dnn']

x_A = Pann_SIG_AGN['effr_frac']
y_A = Pann_SIG_AGN['dnn']

plt.scatter(x, y, color='blue', marker='o', label='Pann_SIG')
plt.scatter(x_A, y_A, color='r', marker='o', label='Pann_SIG_AGN' )


plt.ylabel('Distance from nearest neighbour')
plt.xlabel('Effective radius')
plt.title('Dnn vs eff radius  for isolated galaxies in voids')
plt.legend()
plt.grid(True)

plt.show
